# KOVA3 tutorial 3: query a gene panel with Amazon Athena

**KOVA3 release:** `v3.0.0` (pinned; see the release manifest)
**Tier:** Open (the data are public; Athena runs in *your* AWS account)
**Tools:** an AWS account, `boto3`, and permission to run Athena queries

Tutorial 1 streams one interval out of the VCF. That is the right tool for one gene. For a panel of a few hundred genes, or a list of a few thousand variants, streaming one interval at a time is the wrong shape: you want one query over the whole callset.

That is what the Parquet layer is for. It is columnar and partitioned, so Athena reads only the partitions your query touches and only the columns you name. A gene-panel lookup ends up scanning a few tens of megabytes.

**Who pays for what.** The data are public and the Open Data Sponsorship Program covers their storage and egress. Athena is a service in *your* account: you pay for bytes scanned, currently about USD 5 per TB. The queries below scan megabytes, so a full run of this notebook costs a fraction of a cent. Every cell reports its own bytes scanned so you can see this rather than take it on trust.

> **Draft note.** The `kova3-open` bucket does not exist yet, so these cells carry no outputs. They will be run and the notebook republished with outputs at release.

## 0. Parameters

`ATHENA_OUTPUT` must be a bucket **in your own account** that you can write to. Athena stores query results there; it will not write to the public bucket.

In [ ]:
import boto3, time

BUCKET   = "kova3-open"
REGION   = "ap-northeast-2"
RELEASE  = "v3.0.0"

DATABASE = "kova3"
TABLE    = "sites_v3_0_0"

# CHANGE THIS to a bucket in your own account
ATHENA_OUTPUT = "s3://your-own-bucket/athena-results/"

athena = boto3.client("athena", region_name=REGION)
print(f"querying {DATABASE}.{TABLE} in {REGION}; results go to {ATHENA_OUTPUT}")

## 1. A helper that runs a query and reports what it cost

Athena is asynchronous: you start a query, poll until it finishes, then read the results. The helper also pulls `DataScannedInBytes` out of the execution statistics, which is the number your bill is computed from.

In [ ]:
PRICE_PER_TB = 5.0   # USD, Athena on-demand; check current pricing for your region


def run_query(sql, database=None, quiet=False):
    """Run one Athena query, wait for it, and return (rows, bytes_scanned)."""
    kwargs = {"QueryString": sql,
              "ResultConfiguration": {"OutputLocation": ATHENA_OUTPUT}}
    if database:
        kwargs["QueryExecutionContext"] = {"Database": database}

    qid = athena.start_query_execution(**kwargs)["QueryExecutionId"]

    while True:
        ex = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]
        state = ex["Status"]["State"]
        if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(1)

    if state != "SUCCEEDED":
        raise RuntimeError(ex["Status"].get("StateChangeReason", state))

    scanned = ex["Statistics"]["DataScannedInBytes"]
    if not quiet:
        print(f"scanned {scanned/1e6:.1f} MB  (about ${scanned / 1e12 * PRICE_PER_TB:.6f})")

    rows, token = [], None
    while True:
        page = athena.get_query_results(QueryExecutionId=qid, **({"NextToken": token} if token else {}))
        rows += [[c.get("VarCharValue") for c in r["Data"]] for r in page["ResultSet"]["Rows"]]
        token = page.get("NextToken")
        if not token:
            break
    return rows, scanned

## 2. Register the table, once

You run this once in your account. It creates an external table pointing at the public bucket; no data is copied and nothing is stored on your side.

The important part is **partition projection**. Without it you would have to run a Glue crawler or `MSCK REPAIR TABLE` to discover the roughly 310 partitions, and again after every release. With it, Athena derives the partition paths from the rules in `TBLPROPERTIES` and there is nothing to maintain.

This statement is also published at `metadata/schemas/athena_create_table.sql` in the release bucket, with the release tag substituted, so two releases can be registered side by side as separate tables.

In [ ]:
run_query(f"CREATE DATABASE IF NOT EXISTS {DATABASE}")

CHROMS = ",".join([f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY", "chrM"])

ddl = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {DATABASE}.{TABLE} (
  pos          int,
  ref          string,
  alt          string,
  variant_id   string,
  rsid         string,
  qual         float,
  filter       array<string>,
  ac           int,
  an           int,
  af           double,
  ns           int,
  ns_gt        int,
  ns_nogt      int,
  ns_nodata    int,
  nhomalt      int,
  call_rate    float,
  ic           float,
  hwec2        float,
  hwe          float,
  exchet       float,
  ac_jeju      int,
  an_jeju      int,
  af_jeju      double,
  nhomalt_jeju int
)
PARTITIONED BY (chromosome string, position_bin int)
STORED AS PARQUET
LOCATION 's3://{BUCKET}/data/release={RELEASE}/parquet/'
TBLPROPERTIES (
  'projection.enabled'             = 'true',
  'projection.chromosome.type'     = 'enum',
  'projection.chromosome.values'   = '{CHROMS}',
  'projection.position_bin.type'   = 'integer',
  'projection.position_bin.range'  = '0,24',
  'projection.position_bin.digits' = '3',
  'storage.location.template'      =
    's3://{BUCKET}/data/release={RELEASE}/parquet/chromosome=${{chromosome}}/position_bin=${{position_bin}}/'
)
"""

run_query(ddl, database=DATABASE)
print("table registered")

`rsid` is in the schema but is null throughout this release: KOVA3 does not assign dbSNP identifiers. The stable key is `variant_id`, formatted `chrom-pos-ref-alt`.

## 3. One gene

Start with the same interval tutorial 1 streamed from the VCF, so the two are comparable. *BRCA1* is `chr17:43,044,295-43,125,364` on GRCh38.

**Always constrain `chromosome`.** It is a partition key, and leaving it out makes Athena read every chromosome. Constraining `pos` as well lets the 10 Mb `position_bin` projection prune down to the one or two blocks that overlap the gene.

In [ ]:
BRCA1 = ("chr17", 43044295, 43125364)

sql = f"""
SELECT variant_id, ref, alt, ac, an, af, nhomalt, call_rate
FROM   {DATABASE}.{TABLE}
WHERE  chromosome = '{BRCA1[0]}'
  AND  position_bin BETWEEN {BRCA1[1] // 10_000_000} AND {BRCA1[2] // 10_000_000}
  AND  pos BETWEEN {BRCA1[1]} AND {BRCA1[2]}
ORDER BY pos
"""

rows, _ = run_query(sql, database=DATABASE)
print(f"{len(rows) - 1} variants in BRCA1\n")
for r in rows[:11]:
    print("\t".join("" if v is None else v for v in r))

## 4. A gene panel

This is where Athena earns its place. The same lookup over a panel of genes would be one `bcftools` call per gene; here it is one query.

Give the intervals to Athena as a small inline table and join against it, keeping the partition predicates on the outer query so pruning still happens.

> **Take the intervals from your own annotation source.** The panel below is a three-gene illustration, and the coordinates are the GRCh38 gene bodies. For real work, load your panel from a BED exported from the same Ensembl or RefSeq release your pipeline uses, as shown in the cell after next. An interval that is off by a few kilobases, or taken from a different assembly, returns a confident and wrong answer: the query succeeds, the counts are plausible, and nothing tells you the window was in the wrong place.


In [ ]:
# Illustrative panel only. GRCh38 gene bodies for three genes whose coordinates
# are quoted consistently across the repository. Replace with your own panel,
# loaded from a BED, before using this for anything real: see the next cell.
PANEL = [
    ("BRCA1", "chr17", 43044295, 43125364),
    ("BRCA2", "chr13", 32315086, 32400268),
    ("TP53",  "chr17",  7668402,  7687550),
]

BIN = 10_000_000   # position_bin width; see docs/schemas.md


def panel_query(panel):
    """Build a partition-pruned Athena query for a list of (gene, chrom, start, end)."""
    values = ",\n    ".join(f"('{g}', '{c}', {s}, {e})" for g, c, s, e in panel)
    chroms = ",".join(sorted({f"'{c}'" for _, c, _, _ in panel}))
    bins = sorted({b for _, _, s, e in panel for b in range(s // BIN, e // BIN + 1)})
    return f"""
WITH panel(gene, chrom, start_pos, end_pos) AS (
  VALUES
    {values}
)
SELECT   p.gene,
         count(*)                                 AS n_variants,
         count_if(v.af >= 0.01)                   AS n_common,
         count_if(v.af  < 0.001 AND v.an > 15000) AS n_rare_well_powered,
         round(avg(v.call_rate), 4)               AS mean_call_rate
FROM     {DATABASE}.{TABLE} v
JOIN     panel p
  ON     v.chromosome = p.chrom
 AND     v.pos BETWEEN p.start_pos AND p.end_pos
WHERE    v.chromosome IN ({chroms})
  AND    v.position_bin IN ({",".join(str(b) for b in bins)})
GROUP BY p.gene
ORDER BY n_variants DESC
"""


rows, _ = run_query(panel_query(PANEL), database=DATABASE)
for r in rows:
    print("\t".join("" if v is None else v for v in r))

### Loading a real panel from a BED

For anything beyond a demonstration, take the intervals from the same annotation release your pipeline already uses. A four-column BED (`chrom`, `start`, `end`, `name`) drops straight into `panel_query`.

BED is half-open and zero-based; the `pos` column in KOVA3 is one-based, like the VCF it comes from. Adding one to the start is the whole conversion, and forgetting it quietly drops the first base of every interval.


In [ ]:
PANEL_BED = None   # e.g. "hereditary_cancer_grch38.bed"


def panel_from_bed(path):
    """Read a 4-column BED into the (gene, chrom, start, end) shape panel_query wants.

    BED is zero-based half-open, KOVA3 positions are one-based, so start + 1.
    """
    panel = []
    with open(path) as fh:
        for line in fh:
            if not line.strip() or line.startswith(("#", "track", "browser")):
                continue
            f = line.split()
            chrom, start, end = f[0], int(f[1]), int(f[2])
            name = f[3] if len(f) > 3 else f"{chrom}:{start}-{end}"
            panel.append((name, chrom, start + 1, end))
    return panel


if PANEL_BED:
    panel = panel_from_bed(PANEL_BED)
    print(f"{len(panel)} intervals from {PANEL_BED}")
    rows, scanned = run_query(panel_query(panel), database=DATABASE)
    for r in rows[:20]:
        print("\t".join("" if v is None else v for v in r))
else:
    print("Set PANEL_BED to a BED file to run your own panel.")

## 5. A variant list

The other common shape: you have a list of variants from a clinical report or a published study, and you want the Korean frequency for each. Match on `variant_id`, which is `chrom-pos-ref-alt` and stable across releases.

Supplying the partition predicates alongside the `IN` list is what keeps this cheap. Without them Athena scans the whole callset to find a handful of rows.

In [ ]:
WANTED = [
    "chr17-43093464-A-G",
    "chr13-32340301-G-A",
    "chr17-7674220-C-T",
    "chr11-108267057-G-A",
]

ids = ",".join(f"'{v}'" for v in WANTED)
chroms = ",".join(sorted({f"'{v.split('-')[0]}'" for v in WANTED}))
bins = sorted({int(v.split("-")[1]) // 10_000_000 for v in WANTED})

sql = f"""
SELECT variant_id, ac, an, af, nhomalt, call_rate, af_jeju, an_jeju
FROM   {DATABASE}.{TABLE}
WHERE  chromosome IN ({chroms})
  AND  position_bin IN ({",".join(str(b) for b in bins)})
  AND  variant_id IN ({ids})
"""

rows, _ = run_query(sql, database=DATABASE)
print("\t".join(rows[0]) if rows else "no header")
for r in rows[1:]:
    print("\t".join("" if v is None else v for v in r))

found = {r[0] for r in rows[1:]}
missing = [v for v in WANTED if v not in found]
if missing:
    print(f"\nnot in the callset: {missing}")
    print("Absence here is not evidence of absence in Koreans. Check the callability "
          "resources for these positions before concluding anything.")

## 6. What partition pruning is worth

Worth measuring once, because the difference is the whole argument for the Parquet layer. The same question, asked with and without the partition predicates.

Run this once and then stop: the unpruned version is the expensive one.

In [ ]:
pruned = f"""
SELECT count(*) FROM {DATABASE}.{TABLE}
WHERE chromosome = 'chr17' AND position_bin = 4
  AND pos BETWEEN 43044295 AND 43125364
"""

unpruned = f"""
SELECT count(*) FROM {DATABASE}.{TABLE}
WHERE pos BETWEEN 43044295 AND 43125364
"""

_, a = run_query(pruned, database=DATABASE, quiet=True)
_, b = run_query(unpruned, database=DATABASE, quiet=True)

print(f"with partition predicates:    {a/1e6:>10.1f} MB   ${a/1e12*PRICE_PER_TB:.6f}")
print(f"without partition predicates: {b/1e6:>10.1f} MB   ${b/1e12*PRICE_PER_TB:.6f}")
if a:
    print(f"ratio: {b/a:.0f}x")

The unpruned query also returns the *wrong* answer, not just an expensive one: without a `chromosome` predicate it counts positions in that range on every contig.

## Expected output (to be filled at release)

| Item | Value |
|---|---|
| Release | `v3.0.0` |
| Single gene (*BRCA1*) | `<N>` variants, `<~X>` MB scanned |
| 16-gene panel | `<N>` variants, `<~X>` MB scanned |
| Variant list of 4 | `<N>` found, `<~X>` MB scanned |
| Runtime, whole notebook | `<~1-2 min>`, mostly Athena queue time |
| User-side AWS cost | Athena scan charges in your own account, a fraction of a cent for this notebook |

## Next steps

For genome-wide work the Parquet layer stops being the right tool and the prebuilt Hail Table takes over; that is a planned tutorial. See the [tutorials index](README.md), and [docs/schemas.md](../docs/schemas.md) for the full column list.